<a href="https://colab.research.google.com/github/eniompw/TinyLM/blob/main/TorchMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [3]:
from datasets import load_dataset
import warnings, itertools
warnings.filterwarnings('ignore')

def load_tinystories(num_stories=200, context_size=4):
    """
    Fetches the TinyStories dataset and prepares it for a character-level language model.
    """
    # Fetch data
    dataset = load_dataset('karpathy/tinystories-gpt4-clean', split='train', streaming=True)
    text = ''.join(s['text'] for s in itertools.islice(dataset, num_stories))

    # Build vocabulary and tokenize
    vocab = sorted(set(text))                                       # ordered list of unique characters
    char_to_id = {c: i for i, c in enumerate(vocab)}                # dictionary mapping char to integer id
    encoded = [char_to_id[c] for c in text]                         # map entire text to integer sequence

    # If context_size=1, skip building windows to save RAM (training loop handles slicing).
    if context_size == 1:
        return [], [], vocab, encoded

    # Create sliding windows for inputs and targets using standard Python lists
    inputs = [encoded[i:i+context_size] for i in range(len(encoded)-context_size)] # sliding windows
    targets = encoded[context_size:]                                               # next char to predict

    return inputs, targets, vocab, encoded

In [6]:
import time, torch
import torch.nn as nn, torch.nn.functional as F

# Automatically create all tensors on GPU if available
torch.set_default_device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Parameters ---
context_size = 8  # Increased from 4 for better word coherence
embed_dim = 256
hidden_dim = 200

# --- Data & Tokenization ---
input_ids, target_ids, idx_to_char, token_ids = load_tinystories(num_stories=200, context_size=context_size)
input_ids, target_ids = torch.tensor(input_ids), torch.tensor(target_ids)

# --- Model ---
torch.manual_seed(42)
embedding = nn.Embedding(len(idx_to_char), embed_dim)
mlp = nn.Sequential(
    nn.Linear(context_size * embed_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, len(idx_to_char))
)
optimizer = torch.optim.SGD(list(embedding.parameters()) + list(mlp.parameters()), lr=0.5)

# --- Train ---
batch_size = 1024
start = time.time()
for step in range(3001): # Increased steps for larger context
    batch_idx = torch.randint(0, len(input_ids), (batch_size,))
    x = embedding(input_ids[batch_idx]).view(batch_size, -1)
    loss = F.cross_entropy(mlp(x), target_ids[batch_idx])
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if step % 500 == 0:
        print(f"Step {step:4d} | Loss: {loss:.4f}")

print(f"Training time: {time.time() - start:.1f}s")

# --- Generate ---
@torch.no_grad()
def generate(num_chars=200, context_ids=None):
    if context_ids is None:
        context_ids = list(token_ids[:context_size])
    output_chars = [idx_to_char[i] for i in context_ids]
    for _ in range(num_chars):
        x = embedding(torch.tensor([context_ids])).view(1, -1)
        next_token_probs = torch.softmax(mlp(x), 1)
        next_token = torch.multinomial(next_token_probs, 1).item()
        context_ids, output_chars = context_ids[1:] + [next_token], output_chars + [idx_to_char[next_token]]
    return ''.join(output_chars)

print("\nGenerated Story:\n" + "-"*15)
print(generate())

Step    0 | Loss: 4.1624
Step  500 | Loss: 1.2821
Step 1000 | Loss: 1.1652
Step 1500 | Loss: 1.0054
Step 2000 | Loss: 1.0461
Step 2500 | Loss: 0.9174
Step 3000 | Loss: 0.9474
Training time: 5.9s

Generated Story:
---------------
Once there chess. Dight and laughed and halls.The and Lily, "Saraway," Cakey sche pinnies, and even making and a vear and helped the lady and play up and ratine.
One day, a loved to play it and saw a minside.
